# AFTER: Audio Features Transfer and Exploration in Real-time

**AFTER** is a diffusion-based generative model that creates new audio by blending two sources: one audio stream to set the style or timbre, and another input (either audio or MIDI) to shape the structure over time.

**_Note:_ This notebook is for training an AFTER V2 Autoencoder model.**

Sourcecode: https://github.com/acids-ircam/AFTER

-----

Notebook author: [Martin Heinze](https://codeberg.org/martstilde)

Last updated: 26.07.2026

## Setup

In [ ]:
#Install AFTER

!mkdir /kaggle/temp
%cd /kaggle/temp
!git clone https://github.com/acids-ircam/AFTER
%cd /kaggle/temp/AFTER
!pip install -e .

## Dataset preparation (Autoencoder)

In [ ]:
#Dataset preparation for autoencoder training

!after prepare_dataset \
--input_path /path/to/your/audio-data \
--output_path /kaggle/temp/processed \
--save_waveform True \
--gpu 0

## Copy data (only resume, disabled for initial run)

In [ ]:
##Copy previous run files to /kaggle/working folder to continue training from an earlier checkpoint.
#!cp -r /path/to/your/earlier/runs/* /kaggle/working

## Training (Autoencoder)

In [ ]:
#Autoencoder training
#Select Kaggle's GPU T4 x2 accelerator in your notebook settings.
#If you need to set up a different config (e.g. to change `steps_save` parameter), upload it to a Kaggle dataset and add path after `--config` flag

!CUDA_VISIBLE_DEVICES=0,1 torchrun --nproc_per_node=2  after_scripts/train_autoencoder.py \
--ddp \
--name your-training-name \
--db_path /kaggle/temp/processed \
--config AE_4096 \
--out_path /kaggle/working/ae-runs


##To resume training from earlier checkpoints (replace step after `--restart` flag) use the following command. 
##Phase 2 training usually runs into OOM on batch size > 4, hence the additional flag

#!CUDA_VISIBLE_DEVICES=0,1 torchrun --nproc_per_node=2  after_scripts/train_autoencoder.py \
#--ddp \
#--name your-training-name \
#--db_path /kaggle/temp/processed \
#--config AE_4096 \
#--out_path /kaggle/working/ae-runs
#--restart 100000 \ 
#--batch_size 4

## Export (Autoencoder)

In [ ]:
#Autoencoder model export

!after export_autoencoder \
--model_path /kaggle/working/ae-runs/your-training-name \
--step 500000